## Cell 1 - Library Imports and Dataset Path

- `Path` provides a clear, platform-friendly way to define the dataset directory, and `pandas` is used to load and inspect tabular CSV data. `numpy` is imported for numerical work in the project.
- `DATA_PATH` points to `02_Datasets`, and the `olist_*.csv` pattern discovers the Olist CSV files used in this notebook. The current search finds eight matching datasets; the separate category-translation file does not match this pattern.
- Listing the files confirms which source tables are available before analysis begins.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_PATH = Path("../02_Datasets")
files = sorted(DATA_PATH.glob("olist_*.csv"))
print(f"Total Files: {len(files)}\n")

for file in files:
    print(file.name)

Total Files: 8

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv


## Cell 2 - Dataset Structure Inspection

Each matching CSV is loaded in turn so the notebook can inspect its dimensions, column names, and inferred data types. This first overview helps identify what information is available and which fields may need type conversion before analysis.

**Display note:** the current print statement labels `df.shape[0]` as columns and `df.shape[1]` as rows, so those two displayed labels are reversed; `shape[0]` is the row count and `shape[1]` is the column count.


In [2]:
for file in files:
    df=pd.read_csv(file)

    print(f"\n{"="*60}")
    print(f"File: {file.name}")
    print(f"Columns: {df.shape[0]} | Rows: {df.shape[1]}")
    print(f"\nColumn names:")
    print(df.columns.tolist())
    print(f"\nData types:")
    print(df.dtypes)


File: olist_customers_dataset.csv
Columns: 99441 | Rows: 5

Column names:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Data types:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

File: olist_geolocation_dataset.csv
Columns: 1000163 | Rows: 5

Column names:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Data types:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

File: olist_order_items_dataset.csv
Columns: 112650 | Rows: 7

Column names:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Data types:
order_id     

## Cell 3 - Initial Data Quality Checks

For each dataset, this check counts missing values by column and counts fully duplicated rows. These quick checks help flag potential data-quality issues for later investigation; a flagged value is not by itself a reason to delete a row.


In [3]:
for file in files:
    df=pd.read_csv(file)

    print(f"\n---{file.name}---")
    
    print(f"The missing values:")
    print(df.isnull().sum()[df.isnull().sum()>0])

    print("Duplicated rows: ",df.duplicated().sum())


---olist_customers_dataset.csv---
The missing values:
Series([], dtype: int64)
Duplicated rows:  0

---olist_geolocation_dataset.csv---
The missing values:
Series([], dtype: int64)
Duplicated rows:  261831

---olist_order_items_dataset.csv---
The missing values:
Series([], dtype: int64)
Duplicated rows:  0

---olist_order_payments_dataset.csv---
The missing values:
Series([], dtype: int64)
Duplicated rows:  0

---olist_order_reviews_dataset.csv---
The missing values:
review_comment_title      87656
review_comment_message    58247
dtype: int64
Duplicated rows:  0

---olist_orders_dataset.csv---
The missing values:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
Duplicated rows:  0

---olist_products_dataset.csv---
The missing values:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product

## Cell 4 - Missing-Value Analysis

For columns with missing data, this step calculates both the missing-value count and the percentage of the dataset affected. The counts show the scale of the gaps, while percentages make it easier to compare fields across tables of different sizes. Missing values should be interpreted according to their business meaning rather than automatically deleted.


In [4]:
for file in files:
    df = pd.read_csv(file)
    missing = df.isnull().sum()[df.isnull().sum()>0]
    
    if not missing.empty:
        print(f"\n--- {file.name} ---")
        print(pd.DataFrame({
            "missing_count": missing,
            "missing_percent": (missing / len(df) * 100).round(2)
        }))


--- olist_order_reviews_dataset.csv ---
                        missing_count  missing_percent
review_comment_title            87656            88.34
review_comment_message          58247            58.70

--- olist_orders_dataset.csv ---
                               missing_count  missing_percent
order_approved_at                        160             0.16
order_delivered_carrier_date            1783             1.79
order_delivered_customer_date           2965             2.98

--- olist_products_dataset.csv ---
                            missing_count  missing_percent
product_category_name                 610             1.85
product_name_lenght                   610             1.85
product_description_lenght            610             1.85
product_photos_qty                    610             1.85
product_weight_g                        2             0.01
product_length_cm                       2             0.01
product_height_cm                       2             0.01
prod

## Cells 5-8 - Investigation of Missing Order Approval Timestamps

This step loads the orders table and counts records with a missing `order_approved_at` value by `order_status`. The breakdown is 141 cancelled orders (status `canceled`), 5 created orders, and 14 delivered orders. The delivered records are unusual, so the following cells inspect them rather than assuming the missing timestamp makes them invalid.


In [5]:
orders=pd.read_csv(DATA_PATH/"olist_orders_dataset.csv")

missing_approval = orders[orders["order_approved_at"].isna()]
missing_approval["order_status"].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

### Investigation step - Inspect delivered orders

The 14 delivered orders with missing approval timestamps are displayed alongside their purchase, carrier-handoff, and customer-delivery timestamps. Reviewing the available timeline helps investigate the anomaly without filling in an approval date that is not present in the source data.


In [6]:
missing_approval[
    missing_approval["order_status"] == "delivered"
][[
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]]

,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,2017-02-23 12:04:47,2017-03-01 13:25:33
16567,8a9adc69528e1001fc68dd0aaebbb54a,delivered,2017-02-18 12:45:31,2017-02-23 09:01:52,2017-03-02 10:05:06
19031,7013bcfc1c97fe719a7b5e05e61c12db,delivered,2017-02-18 13:29:47,2017-02-22 16:25:25,2017-03-01 08:07:38
22663,5cf925b116421afa85ee25e99b4c34fb,delivered,2017-02-18 16:48:35,2017-02-22 11:23:10,2017-03-09 07:28:47
23156,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,2017-02-17 13:05:55,2017-02-22 11:23:11,2017-03-02 11:09:19
26800,c1d4211b3dae76144deccd6c74144a88,delivered,2017-01-19 12:48:08,2017-01-25 14:56:50,2017-01-30 18:16:01
38290,d69e5d356402adc8cf17e08b5033acfb,delivered,2017-02-19 01:28:47,2017-02-23 03:11:48,2017-03-02 03:41:58
39334,d77031d6a3c8a52f019764e68f211c69,delivered,2017-02-18 11:04:19,2017-02-23 07:23:36,2017-03-02 16:15:23
48401,7002a78c79c519ac54022d4f8a65e6e8,delivered,2017-01-19 22:26:59,2017-01-27 11:08:05,2017-02-06 14:22:19
61743,2eecb0d85f281280f79fa00f9cec1a95,delivered,2017-02-17 17:21:55,2017-02-22 11:42:51,2017-03-03 12:16:03


### Investigation step - Convert dates for duration analysis

The purchase and customer-delivery fields are converted to datetime values before subtraction. This is necessary because delivery duration is a time interval, not a string comparison.


In [7]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])

### Investigation step - Summarize delivery duration

For the 14 delivered orders, purchase-to-delivery duration has a mean of approximately **12.71 days**, a median of **12 days**, a minimum of **10 days**, and a maximum of **18 days**. These durations do not explain why the approval timestamps are missing. The decision is to retain the orders, preserve the missing timestamps, and document the anomaly - not to invent dates or delete records.


In [8]:
delivered_missing_approval = orders[
    (orders["order_approved_at"].isna()) &
    (orders["order_status"] == "delivered")
].copy() 

delivery_days = (
    delivered_missing_approval["order_delivered_customer_date"]
    - delivered_missing_approval["order_purchase_timestamp"]
).dt.days

delivery_days.describe()

count    14.000000
mean     12.714286
std       2.554892
min      10.000000
25%      11.000000
50%      12.000000
75%      13.000000
max      18.000000
dtype: float64

## Cell 9 - Load Analysis Datasets

This step loads customers, geolocation, order items, payments, reviews, products, and sellers; the orders table was already loaded for the timestamp investigation. Together, these tables support different business questions: who purchased, what was ordered, who sold it, how it was paid for, how customers reviewed it, and where customers and sellers are located. Keeping the existing orders object also preserves its earlier datetime conversions.


In [9]:
customers = pd.read_csv(DATA_PATH / "olist_customers_dataset.csv")
geolocation = pd.read_csv(DATA_PATH / "olist_geolocation_dataset.csv")
order_items = pd.read_csv(DATA_PATH / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_PATH / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_PATH / "olist_order_reviews_dataset.csv")
#orders dataset is already loaded and reloading it disables the changes we made like (text->date format)
products = pd.read_csv(DATA_PATH / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_PATH / "olist_sellers_dataset.csv")

## Cell 10 - Check Order and Order-Item Relationships

The first check examines whether order, customer, product, and seller identifiers are unique in their respective tables, and counts how many distinct orders appear among order items. An order ID can repeat in the order-items table because one order may contain several item rows; it is not expected to be unique there.


In [10]:
print("Orders:", orders["order_id"].is_unique)
print("Customers:", customers["customer_id"].is_unique)
print("Order items:", order_items["order_id"].nunique(), "unique orders")
print("Products:", products["product_id"].is_unique)
print("Sellers:", sellers["seller_id"].is_unique)

Orders: True
Customers: True
Order items: 98666 unique orders
Products: True
Sellers: True


### Cell 10 - Validate Order References

This check looks for order-item rows whose `order_id` is absent from the orders table. The current result is **0 unmatched rows**, supporting the order-to-item relationship check for these records.


In [11]:
unmatched_orders = order_items[
    ~order_items["order_id"].isin(orders["order_id"])
]

print("Unmatched order-item rows:", len(unmatched_orders))

Unmatched order-item rows: 0
 0


## Cell 11 - Validate Product References

Each order-item `product_id` is checked against the products table. This verifies that item records point to known products; the current check finds **0 unmatched product references**.


In [12]:
unmatched_products = order_items[
    ~order_items["product_id"].isin(products["product_id"])
]

print("Unmatched product rows:", len(unmatched_products))

Unmatched product rows: 0


## Cell 12 - Validate Seller References

Each order-item `seller_id` is checked against the sellers table to confirm that the referenced seller exists. The current check finds **0 unmatched seller references**.


In [13]:
unmatched_sellers = order_items[
    ~order_items["seller_id"].isin(sellers["seller_id"])
]

print("Unmatched seller rows:", len(unmatched_sellers))

Unmatched seller rows: 0


## Cell 13 - Validate Customer References

Each order `customer_id` is checked against the customers table. This helps confirm that the customer side of the order relationship is represented; the current check finds **0 unmatched customer references**.


In [14]:
unmatched_customers = orders[
    ~orders["customer_id"].isin(customers["customer_id"])
]

print("Unmatched customer rows:", len(unmatched_customers))

Unmatched customer rows: 0


## Cell 14 - Validate Payment and Review References

Payment and review records are checked separately to confirm that each referenced `order_id` exists in the orders table. The current checks find **0 unmatched payment rows** and **0 unmatched review rows**.


In [15]:
unmatched_payments = payments[
    ~payments["order_id"].isin(orders["order_id"])
]

unmatched_reviews = reviews[
    ~reviews["order_id"].isin(orders["order_id"])
]

print("Unmatched payment rows:", len(unmatched_payments))
print("Unmatched review rows:", len(unmatched_reviews))

Unmatched payment rows: 0
Unmatched review rows: 0


## Cell 15 - Validate Order-Item Uniqueness

The pair (`order_id`, `order_item_id`) is checked as a composite key because an item position is meaningful within its order, not across all orders. The current check finds **0 duplicate pairs**.


In [16]:
print(
    "Duplicate order-item rows:",
    order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

Duplicate order-item rows: 0


## Cell 16 - Geolocation Data Quality

This step checks missing ZIP-code prefixes, latitudes, and longitudes, and counts distinct ZIP prefixes. The current data has **0 missing values** in those three fields and **19,015 unique ZIP prefixes**. Multiple geolocation records can belong to one ZIP prefix, so repeated prefixes are not automatically erroneous and records should not be blindly removed.


In [17]:
print("Missing ZIP prefixes:", geolocation["geolocation_zip_code_prefix"].isna().sum())
print("Missing latitudes:", geolocation["geolocation_lat"].isna().sum())
print("Missing longitudes:", geolocation["geolocation_lng"].isna().sum())

print(
    "Unique ZIP prefixes:",
    geolocation["geolocation_zip_code_prefix"].nunique()
)

Missing ZIP prefixes: 0
Missing latitudes: 0
Missing longitudes: 0
Unique ZIP prefixes: 19015


## Cell 17 - Prepare Order Timestamps

The purchase, approval, carrier-handoff, customer-delivery, and estimated-delivery fields are converted to datetime format. Consistent date types enable monthly trend analysis, delivery-duration calculations, and comparisons between estimated and actual delivery dates. Unparseable values are coerced to missing instead of being replaced with invented dates.


In [18]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

## Current Progress and Next Steps

Initial dataset inspection, selected missing-timestamp investigations, and dataset relationship checks have been performed. These steps do not complete the ETL process: comprehensive cleaning and transformation are still outstanding, and additional transformations may be needed as analysis requirements become clear.

The next notebook will perform exploratory data analysis to address the **17 historical business questions**, followed by predictive modelling for the **9 predictive questions**.
